# 00 — Chuẩn bị dữ liệu cho fine-tune

Đọc dataset trắc nghiệm Nguyên hàm – Tích phân (chỉ các câu `usable`), chia **train/val/test = 80/5/15**
và lưu lên Drive. Hai thí nghiệm (Qwen3-8B và Qwen3-14B) đọc **cùng** thư mục này nên so sánh công bằng.

Cách chia:
- Gom các câu **cùng một bài** (chép lại, sửa lời dẫn) thành nhóm; cả nhóm vào cùng một tập, để test không chứa bản sao của câu train.
- Phân tầng theo (chủ đề, mức độ), seed cố định.

Notebook này chỉ cần CPU.

In [ ]:
#@title 2. Cấu hình
SOURCE = 'trantrien1/vi-math12-integral-mcq'  #@param {type:'string'}
#@markdown ↑ repo dataset trên Hugging Face, hoặc đường dẫn tới `questions.jsonl` (vd. trên Drive)
REPO = 'https://github.com/trantrien1/AQG.git'  #@param {type:'string'}
BRANCH = 'lora-finetune'  #@param {type:'string'}
DRIVE_ROOT = '/content/drive/MyDrive/AQG_ft'  #@param {type:'string'}
DATA_DIR = f'{DRIVE_ROOT}/data'

In [ ]:
#@title 3. Drive, token Hugging Face, hàm chạy lệnh
import os, sys, json, time, shutil, subprocess
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_ROOT, exist_ok=True)

try:  # Colab: 🔑 Secrets -> thêm HF_TOKEN (cần quyền đọc dataset riêng tư)
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception as exc:
    print('Chưa đọc được HF_TOKEN từ Colab Secrets:', exc)

REPO_DIR = '/content/AQG'
NB_DIR = f'{REPO_DIR}/API/notebooks'
ENV = dict(os.environ, PYTHONPATH=NB_DIR, TOKENIZERS_PARALLELISM='false',
           PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')

def sh(cmd, log=None):
    """Chạy lệnh, in đầu ra ngay khi có; lỗi thì dừng notebook."""
    print('$', cmd, flush=True)
    fh = open(log, 'a', encoding='utf-8') if log else None
    p = subprocess.Popen(cmd, shell=True, cwd=NB_DIR if os.path.isdir(NB_DIR) else None,
                         env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, encoding='utf-8', errors='replace')
    for chunk in iter(lambda: p.stdout.read(512), ''):
        sys.stdout.write(chunk)
        if fh:
            fh.write(chunk)
    if fh:
        fh.close()
    if p.wait() != 0:
        raise RuntimeError(f'Lệnh lỗi (mã {p.returncode}): {cmd}')

In [ ]:
#@title 4. Clone repo
subprocess.run(['rm', '-rf', REPO_DIR], check=True)
sh(f'git clone -q --depth 1 -b {BRANCH} {REPO} {REPO_DIR} && git -C {REPO_DIR} log --oneline -1')
sh('pip -q install -U huggingface_hub pytest && python -m pytest -q ../tests/test_mcqft.py')

In [ ]:
#@title 5. Chia tập và lưu lên Drive
sh(f'python -m mcqft.data --source "{SOURCE}" --out "{DATA_DIR}"')

In [ ]:
#@title 6. Xem mẫu huấn luyện
sys.path.insert(0, NB_DIR)
from mcqft.data import load_split
from mcqft.prompts import build_examples
splits = load_split(DATA_DIR)
ex = build_examples(splits['train'][:1], ['gen', 'solve'])
for e in ex:
    print('=' * 30, e['task'])
    for m in e['messages']:
        print(f"[{m['role']}]\n{m['content']}\n")
    print(f"[assistant — phần tính loss]\n{e['target']}\n")